In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Sayantan Das\\OneDrive\\Desktop\\Projects\\Kidney-Diseases-Classification\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Sayantan Das\\OneDrive\\Desktop\\Projects\\Kidney-Diseases-Classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
import CNNClassifier
import CNNClassifier.constants as c

print(c.__file__)
print(dir(c))

c:\users\sayantan das\onedrive\desktop\projects\kidney-diseases-classification\src\CNNClassifier\constants\__init__.py
['CONFIG_FILE_PATH', 'PARAMS_FILE_PATH', 'Path', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [7]:
from CNNClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from CNNClassifier.utils.common import read_yaml,create_directories

In [8]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath=PARAMS_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)

        create_directories([self.config.artifact_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config=self.config.data_ingestion  

        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )    

        return data_ingestion_config

In [9]:
import os
import zipfile
import gdown
from CNNClassifier import logger
from CNNClassifier.utils.common import get_size

In [10]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        # config = self.config.data_ingestion

    
    def download_file(self)-> str:
        '''
        Fetch data from the url
        '''

        try: 
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")

            file_id = dataset_url.split("/")[-2]
            prefix = 'https://drive.google.com/uc?/export=download&id='
            gdown.download(prefix+file_id,zip_download_dir)

            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")

        except Exception as e:
            raise e
        
    

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [11]:
import os

print(os.getcwd())

c:\Users\Sayantan Das\OneDrive\Desktop\Projects\Kidney-Diseases-Classification


In [12]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent

CONFIG_FILE_PATH = ROOT_DIR / "config" / "config.yaml"
PARAMS_FILE_PATH = ROOT_DIR / "params.yaml"

In [13]:
import os
print(os.getcwd())
print(os.path.exists("config/config.yaml"))

c:\Users\Sayantan Das\OneDrive\Desktop\Projects\Kidney-Diseases-Classification
True


In [14]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-30 00:35:32,963:INFO:common:yaml file: config\config.yaml loaded successfully]
[2026-07-30 00:35:32,964:INFO:common:yaml file: params.yaml loaded successfully]
[2026-07-30 00:35:32,965:INFO:common:created directory at: artifacts]
[2026-07-30 00:35:32,967:INFO:common:created directory at: artifacts/data_ingestion]
[2026-07-30 00:35:32,968:INFO:1091974511:Downloading data from https://drive.google.com/file/d/1LOnCl39VSB_9LJamurM7WR_6GiBmq99m/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1LOnCl39VSB_9LJamurM7WR_6GiBmq99m
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1LOnCl39VSB_9LJamurM7WR_6GiBmq99m&confirm=t&uuid=c6b28e8d-e8bf-48cb-81c7-89848354c371
To: c:\Users\Sayantan Das\OneDrive\Desktop\Projects\Kidney-Diseases-Classification\artifacts\data_ingestion\data.zip
100%|██████████| 46.6M/46.6M [00:12<00:00, 3.66MB/s]

[2026-07-30 00:35:49,084:INFO:1091974511:Downloaded data from https://drive.google.com/file/d/1LOnCl39VSB_9LJamurM7WR_6GiBmq99m/view?usp=sharing into file artifacts/data_ingestion/data.zip]
